# Prompt A/B Diff

**What does the way you phrase a prompt cost you?** Fire two (or more) prompts at one model, with your own key, and see each response, token cost, and latency side by side. The classic lesson: a rambling, unbounded ask uses more tokens, costs more, and runs slower than a tight, structured ask that gets the same answer. About a minute, free, no signup.

This notebook is **pre-filled with a demonstration default**: a rambling ask next to a tight one. Run it as-is and watch the rambling prompt cost more for the same answer. The lesson is the token and cost numbers, not a verdict the tool emits.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.pysolvr.com).

---

> **Free tool, provided as-is, no warranty.** It uses *your own* API key and calls the provider directly. **Your key never leaves your machine / your Colab account and is never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2). Keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O prompt_ab_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/prompt-ab/prompt_ab_core.py
!wget -q -O prices.json       https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/prompt-ab/prices.json
print('ready')

## 2. Add your API key (do NOT type it into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store it *outside* the notebook instead. The cell below only *reads* it.

### On Colab (recommended), use the Secrets panel
1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add a secret** for your model's provider, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for the secret. You only need the ONE provider whose model you use.

### On local Jupyter, environment variable
Set it in your shell *before* launching Jupyter: `export OPENAI_API_KEY=sk-...` then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your key stays on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. Pick a model and your prompts

The cell below is pre-filled with a demonstration: the **same task** (apologize to a customer for a late order) asked two ways, a rambling, unbounded version and a tight, structured one. Run it as-is and compare the token counts, cost, and latency. The rambling prompt tends to use more tokens and cost more for the same answer.

Use the provider's own model-ID string (the tool is model-agnostic). Examples: `gpt-5`, `claude-opus-5-5`, `gemini-4-argon`. To find what exists and what it costs: **[llmprover.pysolvr.com/models](https://llmprover.pysolvr.com/models/)** and **[llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/)**.

To compare **your own** wordings: overwrite `PROMPTS` with 2 to 5 of your own prompts (and `LABELS` to name them).

In [ ]:
MODEL = 'gpt-4o-mini'

PROMPTS = [
    ("I was wondering if you could possibly help me out here, I have this situation where I need to "
     "let a customer know that their order is going to be a little bit late, and I want to be really "
     "nice and warm and apologetic about it, maybe explain that we care a lot, and just generally "
     "make them feel good even though it's late, so could you write something for me that does all "
     "of that in a friendly tone? Thanks so much, really appreciate it."),
    "Write a friendly two-sentence apology to a customer for a late order.",
]

# Optional labels, same order as PROMPTS. Set to None to auto-label 'Prompt 1', 'Prompt 2', ...
LABELS = ['rambling / unbounded', 'tight / structured']

## 4. Run it

The prompts fire concurrently. You get each response, token cost (from the dated price snapshot), and latency, side by side, plus a mechanical summary (the heaviest prompt used Nx the tokens of the lightest and cost Nx as much). **No verdict** -- these are token and cost counts, not a judgment of which prompt is better.

In [ ]:
from prompt_ab_core import prompt_ab_core, format_result

result = prompt_ab_core(model=MODEL, prompts=PROMPTS, labels=LABELS)
print(format_result(result))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.pysolvr.com/tools/). For more advanced features, prompts scored for quality across your workload, the which-layer diagnostic, and drift tracked over time, see [Features](https://llmprover.pysolvr.com/features/?tool=prompt-ab).